In [19]:
import keras
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

In [40]:
#Define metrics
def nse(observed_values, simulated_values):
    return 1.0 - np.sum((observed_values - simulated_values) ** 2) / np.sum((observed_values - observed_values.mean()) ** 2)

def rmse(observed_values, predicted_values):
    return np.sqrt(np.mean((observed_values - predicted_values) ** 2))

def kge(observed_values, simulated_values):
    observed_values = np.asarray(observed_values, dtype=float)
    simulated_values = np.asarray(simulated_values, dtype=float)

    r = np.corrcoef(observed_values, simulated_values)[0, 1]
    alpha = simulated_values.std() / observed_values.std()
    beta = simulated_values.mean() / observed_values.mean()
    
    return 1.0 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2) 

In [ ]:
#Helper functions
input_columns = ['prcp_z', 'tmax_z', 'srad_z','vp_z', 'dayl_z', 'drainage_area_km2_z', 'elevation_m_z', 'slope_m_per_km_z','frac_forest_z', 'mean_precip_mm_day_z', 'mean_pet_mm_day_z']
target_column = 'log_runoff'
sequence_length = 180
batch_size = 256

class ModelRecord:
    def __init__(self, model_name):
        self.model_name = f"{model_name} LSTM"
        self.model = keras.models.load_model(f"../lstm_model/lstm_{model_name}.keras")
        self.history = pd.read_csv(f"../lstm_model/history_{model_name}.csv")

        self.training_df = pd.read_csv("../datacleaning-pipeline/training_df.csv")
        self.testing_df = pd.read_csv("../datacleaning-pipeline/testing_df.csv")

        self.y_mu = float(self.training_df[target_column].values.reshape(-1,1).mean())
        self.y_sd = float(self.training_df[target_column].values.reshape(-1,1).std())

    def generate_training_metrics(self, metrics):
        fig, ax = plt.subplots()
        for metric in metrics:
            ax.plot(self.history[metric], label=str(metric))
        ax.set_title(self.model_name)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Error")
        ax.legend()
        fig.savefig(f"./training-metrics/{self.model_name}_{metrics[0]}.png")
        plt.close(fig)

    def generate_evaluation_metrics(self):
        rows = []

        for gauge_id, gauge in self.testing_df.groupby("gauge_id"):
            x, y = self.gauge_arrays(gauge)
            windows = keras.utils.timeseries_dataset_from_array(
                x, None, sequence_length=sequence_length, batch_size=batch_size, shuffle=False
            )
            predictions_log = self.model.predict(windows, verbose=0).ravel() * self.y_sd + self.y_mu
            observations_log = y[sequence_length - 1:] * self.y_sd + self.y_mu
            dates = gauge.sort_values("date")["date"].to_numpy()[sequence_length - 1:]
            keep = np.isfinite(observations_log)
            rows.append(pd.DataFrame({
                "gauge_id": gauge_id,
                "date": dates[keep],
                "obs_runoff": np.expm1(observations_log[keep]),
                "pred_runoff": np.clip(np.expm1(predictions_log[keep]), 0, None),
            }))
            
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(pd.to_datetime(dates[keep]), np.expm1(observations_log[keep]), label="Observed")
            ax.plot(pd.to_datetime(dates[keep]), np.clip(np.expm1(predictions_log[keep]), 0, None), label="Predicted")
            ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
            plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
            ax.set_xlabel("Date")
            ax.set_ylabel("Runoff mm/day")
            ax.set_title(f"{self.model_name} Predictions")
            ax.legend()
            fig.savefig(f"./evaluation-metrics/{self.model_name}/{gauge_id}_predictions.png")
            plt.close(fig)

        test = pd.concat(rows, ignore_index=True)

        cutoff = self.training_df.groupby("gauge_id")["runoff_mm_day"].quantile(0.95).rename("cutoff")
        test = test.join(cutoff, on="gauge_id")
        test["obs_high"]  = test["obs_runoff"]  > test["cutoff"]
        test["pred_high"] = test["pred_runoff"] > test["cutoff"]

        per_basin_nse = pd.Series({g: nse(d["obs_runoff"].to_numpy(), d["pred_runoff"].to_numpy())
                            for g, d in test.groupby("gauge_id")})
        per_basin_kge = pd.Series({g: kge(d["obs_runoff"].to_numpy(), d["pred_runoff"].to_numpy())
                            for g, d in test.groupby("gauge_id")})
        hits = (test["pred_high"] & test["obs_high"]).sum()

        results = pd.DataFrame({f"{self.model_name}": pd.Series({
            "NSE pooled":           nse(test["obs_runoff"].to_numpy(), test["pred_runoff"].to_numpy()),
            "NSE per-basin median": per_basin_nse.median(),
            "KGE pooled":           kge(test["obs_runoff"].to_numpy(), test["pred_runoff"].to_numpy()),
            "KGE per-masin median": per_basin_kge.median(),
            "RMSE mm/day":          rmse(test["obs_runoff"], test["pred_runoff"]),
            "high-flow recall":     hits / test["obs_high"].sum(),
            "high-flow precision":  hits / max(test["pred_high"].sum(), 1),
        })}).round(3)
        print(f"test rows scored: {len(test):,}   basins: {test['gauge_id'].nunique()}   "
            f"high-flow days: {test['obs_high'].sum():,}")
        
        return results

    def gauge_arrays(self, gauge):
        #Process data by gauge
        gauge = gauge.sort_values('date')
        x = gauge[input_columns].to_numpy(np.float32)
        y = ((gauge[target_column].to_numpy(np.float32) - self.y_mu) / self.y_sd).astype(np.float32)

        #Drop warmup datapoints
        if 'split' in gauge:
            y[(gauge['split'] == 'warmup').to_numpy()] = np.nan
        return x, y
            

In [45]:
#Load the models that need to be evaluated
model_names = ["Base", "256x2", "128x1"]
models = []
eval_metrics = []

for model_name in model_names:
    models.append(ModelRecord(model_name))

#Generate metrics
for model in models:
    model.generate_training_metrics(["loss", "val_loss"])
    model.generate_training_metrics(["r2", "val_r2"])

    eval_metrics.append(model.generate_evaluation_metrics())

joined_eval_metrics = pd.concat(eval_metrics, axis=1)
joined_eval_metrics.to_csv("./evaluation-metrics/eval_metrics.csv")
joined_eval_metrics


test rows scored: 75,194   basins: 82   high-flow days: 3,630
test rows scored: 75,194   basins: 82   high-flow days: 3,630
test rows scored: 75,194   basins: 82   high-flow days: 3,630


,Base LSTM,256x2 LSTM,128x1 LSTM
NSE pooled,0.796,0.785,0.811
NSE per-basin median,0.770,0.786,0.799
KGE pooled,0.867,0.892,0.881
KGE per-masin median,0.798,0.833,0.823
RMSE mm/day,2.761,2.836,2.661
high-flow recall,0.607,0.611,0.626
high-flow precision,0.617,0.659,0.687
